# Sinh dữ liệu giá nhà thô (chưa làm sạch)

Dùng cho Bài 02 – Quy trình thực hiện một dự án Học máy: thực hành làm sạch dữ liệu, chọn và thiết kế đặc trưng, dự án dự đoán giá nhà.

| Cột | Ý nghĩa | Đơn vị / giá trị |
|---|---|---|
| `area` | Diện tích | m² |
| `bedrooms` | Số phòng ngủ | phòng |
| `distance` | Khoảng cách tới trung tâm | km |
| `district` | Khu vực | `center` (trung tâm), `suburb` (ven đô), `outskirts` (ngoại thành) |
| `year_built` | Năm xây dựng | năm |
| `paint_color` | Màu sơn | `white`, `yellow`, `blue`, `gray` |
| `price` | Giá bán (**biến mục tiêu**) | triệu VND |

Quy luật sinh giá (người học không biết trước, phải tự khám phá):

$$y = 300 + 30\,x_{\text{area}} + 150\,x_{\text{bedrooms}} - 40\,x_{\text{distance}} + \text{phụ phí khu vực} - 10 \times \text{tuổi nhà} + \text{nhiễu}$$

với phụ phí khu vực: trung tâm +600, ven đô +200, ngoại thành 0; tuổi nhà = 2026 − năm xây dựng. **Màu sơn không ảnh hưởng đến giá.**

Sau đó cố ý đưa vào các lỗi thường gặp của dữ liệu thực tế: giá trị thiếu, dòng trùng lặp, giá trị ngoại lai do nhập sai, chữ viết không thống nhất.

**Lưu ý:** chạy notebook này sẽ ghi đè file `house_prices_raw.csv`.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)   # cố định bộ sinh số ngẫu nhiên để lần chạy nào cũng ra cùng dữ liệu
N = 1000                            # số căn nhà

## 1. Sinh dữ liệu sạch

In [2]:
area = np.round(rng.uniform(30, 200, N), 1)                     # diện tích (m²)
# Số phòng ngủ phụ thuộc vào diện tích (nhà rộng thường nhiều phòng hơn)
bedrooms = np.clip(np.round(area / 40 + rng.normal(0, 0.7, N)), 1, 6)
distance = np.round(rng.uniform(0.5, 25, N), 1)                 # khoảng cách tới trung tâm (km)
# Khu vực xác định theo khoảng cách: dưới 5 km là trung tâm, dưới 15 km là ven đô, còn lại là ngoại thành
district = np.where(distance < 5, 'center', np.where(distance < 15, 'suburb', 'outskirts'))
year_built = rng.integers(1990, 2026, N)                         # năm xây dựng
paint_color = rng.choice(['white', 'yellow', 'blue', 'gray'], N)  # màu sơn, không ảnh hưởng đến giá

district_premium = pd.Series(district).map({'center': 600, 'suburb': 200, 'outskirts': 0}).values  # phụ phí khu vực
age = 2026 - year_built                                          # tuổi nhà (năm)
noise = rng.normal(0, 250, N)                                    # nhiễu ngẫu nhiên
price = 300 + 30 * area + 150 * bedrooms - 40 * distance + district_premium - 10 * age + noise
price = np.round(np.maximum(price, 300), 0)                      # giá thấp nhất 300 triệu

df = pd.DataFrame({
    'area': area, 'bedrooms': bedrooms, 'distance': distance, 'district': district,
    'year_built': year_built, 'paint_color': paint_color, 'price': price,
})

## 2. Đưa vào các lỗi thường gặp

In [3]:
# Giá trị thiếu: 40 dòng thiếu năm xây dựng, 15 dòng thiếu số phòng ngủ, 5 dòng thiếu giá
df.loc[rng.choice(N, 40, replace=False), 'year_built'] = np.nan
df.loc[rng.choice(N, 15, replace=False), 'bedrooms'] = np.nan
df.loc[rng.choice(N, 5, replace=False), 'price'] = np.nan

# Ngoại lai do nhập sai: 6 dòng diện tích bị gõ thừa một chữ số 0 (nhân 10)
typo_rows = rng.choice(N, 6, replace=False)
df.loc[typo_rows, 'area'] = df.loc[typo_rows, 'area'] * 10

# Chữ viết không thống nhất trong cột khu vực (viết hoa, thừa khoảng trắng)
messy_rows = rng.choice(N, 30, replace=False)
df.loc[messy_rows, 'district'] = df.loc[messy_rows, 'district'].map(
    lambda name: rng.choice([name.upper(), ' ' + name, name.capitalize() + ' ']))

# Dòng trùng lặp: 20 dòng bị nhập hai lần, rồi xáo trộn thứ tự
df = pd.concat([df, df.sample(20, random_state=1)], ignore_index=True)
df = df.sample(frac=1, random_state=2).reset_index(drop=True)

In [4]:
df.to_csv('house_prices_raw.csv', index=False)
print('Đã lưu', len(df), 'dòng vào house_prices_raw.csv')
df.head()

Đã lưu 1020 dòng vào house_prices_raw.csv


,area,bedrooms,distance,district,year_built,paint_color,price
0,173.2,4.0,9.1,suburb,2024.0,gray,5786.0
1,123.5,4.0,8.3,suburb,2011.0,blue,4675.0
2,129.8,3.0,4.6,center,NaN,white,4984.0
3,69.2,1.0,6.2,suburb,1995.0,blue,1902.0
4,110.6,3.0,15.9,outskirts,2011.0,yellow,3222.0
